# 4. LLM Fine-Tuning (Stage 2)

**Objective:** Train the main Language Model (e.g., GPT-2) on the `assorted_train.jsonl` dataset we just created. We will use the Hugging Face `Trainer` for this.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/irene-30/DLAI.git
%cd /content/DLAI

Cloning into 'DLAI'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 229 (delta 108), reused 128 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (229/229), 115.38 KiB | 6.07 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/DLAI


In [ ]:
%pip install datasets transformers torch accelerate

In [ ]:
import sys
import os
import torch
from transformers import Trainer, TrainingArguments

# Add 'src' to path
#sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils import (
    get_llm_tokenizer, MAX_SEQ_LEN, LLM_MODEL_NAME,
    PATH_LLM_MODEL, PATH_PROCESSED_DATA
)
from src.dataset import AssortedDataset
from src.model.transformer import get_llm_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
PATH_PROCESSED_DATA = "/content/drive/My Drive/DLAI/data/processed/assorted_train_E14.jsonl"

## 4.1 Load Tokenizer and Assorted Dataset

In [ ]:
from datasets import load_dataset

tokenizer = get_llm_tokenizer()

try:
    # 1. Carica i dati grezzi (solo testo)
    raw_assorted_data = load_dataset("json", data_files=PATH_PROCESSED_DATA, split="train")

    # 2. Definisci la funzione di tokenizzazione
    def tokenize_function(examples):
        # Tokenizza il testo
        tokenized = tokenizer(
            examples["text"],
            max_length=512,        # MAX_SEQ_LEN
            padding="max_length",
            truncation=True,
        )
        # Imposta le 'labels' per il Causal LM
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    # 3. (LA PARTE IMPORTANTE) Applica la tokenizzazione a tutto il dataset
    # Questo crea le colonne 'input_ids', 'attention_mask' e 'labels'
    tokenized_dataset = raw_assorted_data.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"]  # Rimuove la colonna 'text' originale
    )

    print(f"Loaded and tokenized {len(tokenized_dataset)} assorted samples.")
    print(f"Nuove colonne del Dataset: {tokenized_dataset.column_names}")

except FileNotFoundError:
    print(f"ERROR: Processed data not found at {PATH_PROCESSED_DATA}")
    print("Please run '03_preprocessing_assorted.ipynb' first.")

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Loaded and tokenized 7473 assorted samples.
Nuove colonne del Dataset: ['input_ids', 'attention_mask', 'labels']


## 4.2 Load LLM and Resize Embeddings

In [ ]:
# This helper function loads the model AND resizes its token embeddings
model = get_llm_model(
    model_name=LLM_MODEL_NAME,
    tokenizer_len=len(tokenizer)
).to(device)

print(f"LLM parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

Loading base LLM: gpt2
Resizing model token embeddings from 50257 to 51284
LLM parameters: 125.23M


## 4.3 Set Up Trainer and Run

In [ ]:
# --- Training Configuration ---
NUM_TRAIN_EPOCHS = 5
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-5

# --- 1. Definisci il percorso di salvataggio sul Drive ---
drive_llm_dir = "/content/drive/My Drive/TokenAssorted_GSM8K_Results/llm_stage2_checkpoints/"

training_args = TrainingArguments(
    output_dir=drive_llm_dir,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=20,
    report_to="none",
    fp16=True if device == "cuda" else False,

    # --- Impostazioni per i Checkpoint ---
    save_strategy="steps",         # Salva ogni tot. "steps" (passaggi)
    save_steps=200,                    # Salva un checkpoint ogni 200 passaggi
    save_total_limit=2,                # Tieni solo gli ultimi 2 checkpoint (risparmia spazio)
    eval_strategy="steps",       # Necessario per 'load_best_model_at_end'
    eval_steps=200,                    # Valuta ogni 200 passaggi
    load_best_model_at_end=True,     # Carica il modello migliore alla fine del training
    metric_for_best_model="eval_loss", # Usa la "loss" per decidere qual è il "migliore"
)

# --- Dobbiamo anche dare al Trainer un set di validazione ---
# Ora questo 'train_test_split' funziona perché 'tokenized_dataset'
# ha già le colonne 'input_ids', 'labels', ecc.
train_test_split = tokenized_dataset.train_test_split(test_size=0.1)
train_data_final = train_test_split['train']
eval_data_final = train_test_split['test']


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data_final,   # Usa il 90%
    eval_dataset=eval_data_final,     # Usa il 10% per la validazione
    tokenizer=tokenizer,
)

print("--- Starting LLM Fine-Tuning ---")
trainer.train()

/tmp/ipython-input-452595478.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


--- Starting LLM Fine-Tuning ---


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## 4.4 Save Final Model

Save the final model and tokenizer to the experiments directory.

In [ ]:
PATH_LLM_MODEL = "/content/drive/My Drive/DLAI/experiments/llm_stage2"

In [ ]:
print(f"Saving final LLM to {PATH_LLM_MODEL}")
trainer.save_model(PATH_LLM_MODEL)
tokenizer.save_pretrained(PATH_LLM_MODEL)
print("Final model saved.")